In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# --- 1. CONFIGURATION ---
API_ENDPOINT = "https://analytics.api.aiesec.org/v2/applications/analyze.json"
ACCESS_TOKEN = "YOUR_ACCESS_TOKEN_HERE" # User must insert their GIS Token
MC_INDIA_ID = 1589 # Standard ID for MC India, verify if needed

def fetch_data(start_date, end_date):
    """
    Fetches application data for MC India within the date range.
    Note: In a real scenario, this would likely need to handle pagination
    or be broken down by smaller date chunks if the API limit is hit.
    """
    headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
    params = {
        "start_date": start_date,
        "end_date": end_date,
        "home_office_id": MC_INDIA_ID,
        "type": "person" # Assuming we are counting applicants
    }

    # Simulating the request structure.
    # If the API returns a list of individual apps:
    try:
        response = requests.get(API_ENDPOINT, params=params, headers=headers)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Error fetching data: {e}")
        # FOR DEMO PURPOSES: Returning a mock DataFrame structure if API fails
        # so the code below can still be demonstrated.
        return generate_mock_data()

def generate_mock_data():
    """Generates synthetic data mirroring AIESEC seasonality for demonstration."""
    dates = pd.date_range(start="2022-01-01", end="2025-12-31", freq="D")
    data = []
    for date in dates:
        # Simulate high summer/winter peaks
        base_val = 10
        if date.month in [1, 5, 6, 7, 11, 12]:
            base_val = 50
        count = np.random.poisson(base_val)
        for _ in range(count):
            data.append({"created_at": date.strftime("%Y-%m-%d")})
    return data

# --- 2. DATA PREPARATION ---
raw_data = fetch_data("2022-01-01", "2025-12-31")

# Convert to DataFrame
if isinstance(raw_data, dict) and 'data' in raw_data:
     df = pd.DataFrame(raw_data['data'])
else:
     df = pd.DataFrame(raw_data)

# Preprocessing
df['created_at'] = pd.to_datetime(df['created_at'])
df['month_year'] = df['created_at'].dt.to_period('M')

# Aggregate Data: Count applications per month
monthly_data = df.groupby('month_year').size().reset_index(name='app_count')
monthly_data['month_year'] = monthly_data['month_year'].dt.to_timestamp()

# Feature Engineering for ML
monthly_data['month'] = monthly_data['month_year'].dt.month
monthly_data['year'] = monthly_data['month_year'].dt.year
monthly_data['quarter'] = monthly_data['month_year'].dt.quarter

# --- 3. MODELING (Predicting 2026) ---
# Prepare Training Data
X = monthly_data[['month', 'year', 'quarter']]
y = monthly_data['app_count']

# Train Random Forest Regressor
# Using Random Forest as it handles non-linear seasonality well without complex tuning
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

# Prepare 2026 Prediction Data
future_dates = pd.date_range(start="2026-01-01", end="2026-12-31", freq="MS")
future_df = pd.DataFrame({
    'month_year': future_dates,
    'month': future_dates.month,
    'year': future_dates.year,
    'quarter': future_dates.quarter
})

# Predict
future_df['predicted_apps'] = model.predict(future_df[['month', 'year', 'quarter']])

# Identify High Activity Months (Top 3)
high_activity = future_df.nlargest(3, 'predicted_apps')

# --- 4. VISUALIZATION & OUTPUT ---
plt.figure(figsize=(12, 6))
plt.plot(monthly_data['month_year'], monthly_data['app_count'], label='Historical (2022-2025)')
plt.plot(future_df['month_year'], future_df['predicted_apps'], label='Predicted (2026)', linestyle='--', color='orange')
plt.title('AIESEC MC India: Application Trends & 2026 Prediction')
plt.xlabel('Date')
plt.ylabel('Applications')
plt.legend()
plt.grid(True)
plt.show()

print("\n--- PREDICTION RESULTS FOR 2026 ---")
print(future_df[['month_year', 'predicted_apps']])
print("\n--- HIGH ACTIVITY MONTHS PREDICTION ---")
print(high_activity)

# Save to CSV
final_csv = pd.concat([monthly_data, future_df])
final_csv.to_csv('mc_india_forecast_2026.csv', index=False)
print("\nFile saved as mc_india_forecast_2026.csv")